In [ ]:
# =============================================================================
# TAHAP 0: IMPORT LIBRARY
# =============================================================================
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb  # Ditambahkan untuk blending
import optuna          # Ditambahkan untuk optimisasi
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')
print("✅ Library berhasil diimpor.")

# =============================================================================
# TAHAP 1: PERSIAPAN DATA DASAR
# =============================================================================
print("\n--- Tahap 1: Memuat dan Mempersiapkan Data ---")
# Ganti dengan path ke folder dataset Anda
DATA_PATH = '.../dataset/'

try:
    train_trans_df = pd.read_csv(f'{DATA_PATH}train_transaction_data.csv')
    test_trans_df = pd.read_csv(f'{DATA_PATH}test_transaction_data.csv')
    prodgram_df = pd.read_csv(f'{DATA_PATH}prodgram_data.csv')
    train_label_df = pd.read_csv(f'{DATA_PATH}train_label_data.csv')
    # Membuat kerangka submission dari data tes untuk memastikan semua MemberID ter-cover
    sample_submission_df = pd.read_csv(f'{DATA_PATH}test_transaction_data.csv')[['MemberID']].drop_duplicates()
    print("✅ Semua file berhasil dimuat.")
except FileNotFoundError as e:
    print(f"❌ Error: File tidak ditemukan. Pastikan path '{DATA_PATH}' sudah benar. Detail: {e}")
    exit()

# =============================================================================
# TAHAP 2: REKAYASA FITUR BERBASIS SIKLUS KONSUMSI
# =============================================================================
print("\n--- Tahap 2: Rekayasa Fitur Berbasis Konsumsi ---")

# Ekstrak nilai gramasi numerik dari nama
prodgram_df['GrammageNumeric'] = prodgram_df['GrammageName'].str.extract('(\d+)').astype(float)
prodgram_df['GrammageNumeric'].fillna(prodgram_df['GrammageNumeric'].median(), inplace=True)

# Gabungkan semua data transaksi
all_trans_df = pd.concat([train_trans_df, test_trans_df], ignore_index=True)
all_trans_df['TransactionDatetime'] = pd.to_datetime(all_trans_df['TransactionDatetime']).dt.tz_localize(None)

# Gabungkan dengan data prodgram untuk mendapatkan gramasi
all_trans_df = pd.merge(all_trans_df, prodgram_df[['prodgramID', 'GrammageNumeric']], left_on='FK_PROD_GRAM_ID', right_on='prodgramID', how='left')

# Hitung 'TotalGrammage' sebagai sinyal konsumsi utama
all_trans_df['TotalGrammage'] = all_trans_df['Qty'] * all_trans_df['GrammageNumeric']
all_trans_df['Month'] = all_trans_df['TransactionDatetime'].dt.to_period('M')
print("✅ Fitur 'TotalGrammage' sebagai sinyal konsumsi telah dibuat.")

# Membuat Agregat Bulanan
monthly_agg = all_trans_df.groupby(['MemberID', 'Month']).agg(
    TotalGrammage=('TotalGrammage', 'sum'),
    Frequency=('TransactionID', 'nunique')
).reset_index()
print("✅ Agregat konsumsi & frekuensi bulanan telah dibuat.")

# Membuat "Kanvas" Data Latih (Point-in-Time)
all_months = monthly_agg['Month'].unique()
all_members = all_trans_df['MemberID'].unique()
canvas_df = pd.MultiIndex.from_product([all_members, all_months], names=['MemberID', 'Month']).to_frame(index=False)
data = pd.merge(canvas_df, monthly_agg, on=['MemberID', 'Month'], how='left').fillna(0)
data = data.sort_values(by=['MemberID', 'Month'])

# Membuat Lag & Rolling Features
features_to_lag = ['TotalGrammage', 'Frequency']
lags = [1, 2, 3]
windows = [3, 6]

for feat in features_to_lag:
    for lag in lags:
        data[f'{feat}_lag_{lag}'] = data.groupby('MemberID')[feat].shift(lag).fillna(0)
    for window in windows:
        data[f'{feat}_rolling_mean_{window}'] = data.groupby('MemberID')[feat].shift(1).rolling(window, min_periods=1).mean().fillna(0)
        data[f'{feat}_rolling_std_{window}'] = data.groupby('MemberID')[feat].shift(1).rolling(window, min_periods=1).std().fillna(0)
print("✅ Fitur Lag dan Rolling Window telah dibuat.")

# Membuat Label Target
data['target'] = data.groupby('MemberID')['TotalGrammage'].shift(-1).fillna(0)
data['target'] = (data['target'] > 0).astype(int)
# Hapus bulan terakhir dari setiap member karena tidak memiliki target
data_final = data.groupby('MemberID').head(-1).copy()
print("✅ Label target telah dibuat.")

# =============================================================================
# TAHAP 3: PERSIAPAN DATA LATIH DAN TES FINAL
# =============================================================================
print("\n--- Tahap 3: Mempersiapkan Data Latih dan Tes ---")

# Data Latih: Semua data historis dari member yang ada di train_label_df
train_member_ids = train_label_df['MemberID'].unique()
X_full_train = data_final[data_final['MemberID'].isin(train_member_ids)].copy()
y_full_train = X_full_train['target']

# Data Tes: Fitur dari bulan terakhir untuk setiap member
latest_features_df = data.loc[data.groupby('MemberID')['Month'].idxmax()]
X_test_final = pd.merge(sample_submission_df[['MemberID']], latest_features_df, on='MemberID', how='left').fillna(0)

# Definisikan kolom fitur
features = [col for col in data_final.columns if col not in ['MemberID', 'Month', 'TotalGrammage', 'Frequency', 'target']]
X_full_train = X_full_train[features]
X_test_final = X_test_final[features]
print("✅ Data Latih dan Tes siap untuk pemodelan.")

<>:41: SyntaxWarning: invalid escape sequence '\d'
<>:41: SyntaxWarning: invalid escape sequence '\d'
C:\Users\MSI Modern 14\AppData\Local\Temp\ipykernel_3988\3387442796.py:41: SyntaxWarning: invalid escape sequence '\d'
  prodgram_df['GrammageNumeric'] = prodgram_df['GrammageName'].str.extract('(\d+)').astype(float)


✅ Library berhasil diimpor.

--- Tahap 1: Memuat dan Mempersiapkan Data ---
❌ Error: File tidak ditemukan. Pastikan path './' sudah benar. Detail: [Errno 2] No such file or directory: './train_transaction_data.csv'

--- Tahap 2: Rekayasa Fitur Berbasis Konsumsi ---


NameError: name 'prodgram_df' is not defined

: 